# Phase 03 – Dataset Selection, ML Baseline & erste Evaluation

## 1. Wind Farm A – Event Distribution by Turbine

In [1]:
from google.colab import drive
from pathlib import Path
import pandas as pd

# Mount Google Drive
drive.mount("/content/drive")

# Define dataset directory
data_root = Path("/content/drive/MyDrive/CARE_To_Compare")

# Find Wind Farm A event metadata
metadata_files = [
    path for path in data_root.rglob("event_info.csv")
    if "wind farm a" in str(path.parent).lower().replace("_", " ")
]

if len(metadata_files) != 1:
    raise FileNotFoundError(
        f"Expected one Wind Farm A metadata file, found {len(metadata_files)}"
    )

# Reload event metadata with the correct delimiter
events = pd.read_csv(metadata_files[0], sep=";")

# Count normal and anomaly events per turbine
summary = pd.crosstab(
    events["asset"],
    events["event_label"]
)

summary["total_events"] = summary.sum(axis=1)

print("Wind Farm A – Event Distribution by Turbine")
display(summary)

# Display event details
print("\nWind Farm A – Event Details")

display(
    events[
        ["asset", "event_id", "event_label", "event_description"]
    ].sort_values(["asset", "event_id"])
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Wind Farm A – Event Distribution by Turbine


event_label,anomaly,normal,total_events
asset,,,
0,3,2,5
10,3,2,5
11,1,3,4
13,2,2,4
21,3,1,4



Wind Farm A – Event Details


,asset,event_id,event_label,event_description
4,0,0,anomaly,Generator bearing failure
14,0,24,normal,NaN
5,0,26,anomaly,Hydraulic group
18,0,71,normal,NaN
3,0,73,anomaly,Hydraulic group
15,10,3,normal,NaN
8,10,10,anomaly,Gearbox failure
16,10,17,normal,NaN
6,10,40,anomaly,Generator bearing failure
7,10,42,anomaly,Hydraulic group


## 2. Candidate Turbine – Event Timeline

In [2]:
# Select events for candidate turbine 0
candidate_events = events[
    events["asset"] == 0
].copy()

# Convert event timestamps
candidate_events["event_start"] = pd.to_datetime(
    candidate_events["event_start"],
    errors="coerce"
)

candidate_events["event_end"] = pd.to_datetime(
    candidate_events["event_end"],
    errors="coerce"
)

# Sort events chronologically
candidate_events = candidate_events.sort_values(
    "event_start"
)

# Display event timeline
display(
    candidate_events[
        [
            "event_id",
            "event_label",
            "event_start",
            "event_end",
            "event_description"
        ]
    ]
)

,event_id,event_label,event_start,event_end,event_description
18,71,normal,2023-01-02 00:00:00,2023-01-16 00:00:00,NaN
14,24,normal,2023-04-27 15:00:00,2023-05-11 11:20:00,NaN
3,73,anomaly,2023-06-10 11:40:00,2023-06-17 11:40:00,Hydraulic group
4,0,anomaly,2023-08-06 06:10:00,2023-08-20 06:10:00,Generator bearing failure
5,26,anomaly,2023-10-12 10:20:00,2023-10-19 10:20:00,Hydraulic group


## 3. Normal Event Data Inspection

In [3]:
# Inspect normal event datasets for turbine 0

farm_dir = metadata_files[0].parent

normal_event_ids = [71, 24]

for event_id in normal_event_ids:

    matching_files = list(
        farm_dir.rglob(f"{event_id}.csv")
    )

    if len(matching_files) != 1:
        print(f"Event {event_id}: Expected one CSV file, found {len(matching_files)}")
        continue

    file_path = matching_files[0]

    # Load a small sample and detect the delimiter
    sample = pd.read_csv(
        file_path,
        sep=None,
        engine="python",
        nrows=5
    )

    print(f"\nEvent {event_id}")
    print(f"File: {file_path.name}")
    print(f"Number of columns: {len(sample.columns)}")

    # Identify relevant metadata columns
    relevant_columns = [
        col for col in sample.columns
        if any(
            keyword in col.lower()
            for keyword in [
                "time", "date", "train",
                "test", "pred", "status", "split"
            ]
        )
    ]

    print("Relevant columns:", relevant_columns)

    if relevant_columns:
        display(sample[relevant_columns])
    else:
        display(sample.iloc[:, :8])


Event 71
File: 71.csv
Number of columns: 86
Relevant columns: ['time_stamp', 'train_test', 'status_type_id']


,time_stamp,train_test,status_type_id
0,2022-01-01 00:00:00,train,0
1,2022-01-01 00:10:00,train,0
2,2022-01-01 00:20:00,train,0
3,2022-01-01 00:30:00,train,0
4,2022-01-01 00:40:00,train,0



Event 24
File: 24.csv
Number of columns: 86
Relevant columns: ['time_stamp', 'train_test', 'status_type_id']


,time_stamp,train_test,status_type_id
0,2022-04-24 15:00:00,train,4
1,2022-04-24 15:10:00,train,4
2,2022-04-24 15:20:00,train,4
3,2022-04-24 15:30:00,train,4
4,2022-04-24 15:40:00,train,4


## 4. Training and Prediction Period Analysis

In [4]:
# Analyze training and prediction periods

for event_id in [71, 24]:

    file_path = list(
        farm_dir.rglob(f"{event_id}.csv")
    )[0]

    # Load only the required columns
    df = pd.read_csv(
        file_path,
        sep=None,
        engine="python",
        usecols=["time_stamp", "train_test"]
    )

    # Convert timestamps
    df["time_stamp"] = pd.to_datetime(
        df["time_stamp"]
    )

    # Summarize time periods
    summary = df.groupby("train_test").agg(
        observations=("time_stamp", "size"),
        start=("time_stamp", "min"),
        end=("time_stamp", "max")
    )

    print(f"\nEvent {event_id}")
    display(summary)


Event 71


,observations,start,end
train_test,,,
prediction,2449,2023-01-01,2023-01-18 00:00:00
train,52295,2022-01-01,2022-12-31 23:50:00



Event 24


,observations,start,end
train_test,,,
prediction,2714,2023-04-24 15:00:00,2023-05-13 11:20:00
train,52289,2022-04-24 15:00:00,2023-04-24 14:50:00


## 5. Timestamp Overlap Analysis

In [5]:
# Load timestamps and data splits for both events

def load_event_splits(event_id):
    file_path = list(farm_dir.rglob(f"{event_id}.csv"))[0]

    df = pd.read_csv(
        file_path,
        sep=None,
        engine="python",
        usecols=["time_stamp", "train_test"]
    )

    df["time_stamp"] = pd.to_datetime(df["time_stamp"])

    return {
        split: set(df.loc[df["train_test"] == split, "time_stamp"])
        for split in ["train", "prediction"]
    }


event_71 = load_event_splits(71)
event_24 = load_event_splits(24)

# Check timestamp overlaps
train_overlap = event_71["train"] & event_24["train"]

leakage_overlap = (
    event_71["prediction"] & event_24["train"]
)

print("Shared training timestamps:", len(train_overlap))

print(
    "Event 71 prediction timestamps also in Event 24 training:",
    len(leakage_overlap)
)

print(
    "Total Event 71 prediction timestamps:",
    len(event_71["prediction"])
)

Shared training timestamps: 35760
Event 71 prediction timestamps also in Event 24 training: 2449
Total Event 71 prediction timestamps: 2449


## 6. Sensor Value Comparison Across Events

In [6]:
import numpy as np

# Select sensors for comparison
sensor_cols = [
    "sensor_13_avg",
    "sensor_14_avg",
    "sensor_18_avg"
]

# Load selected columns from both events
def load_sensor_data(event_id, split):

    # Find event CSV recursively
    matching_files = list(
        farm_dir.rglob(f"{event_id}.csv")
    )

    if len(matching_files) != 1:
        raise FileNotFoundError(
            f"Expected one file for Event {event_id}, "
            f"found {len(matching_files)}"
        )

    file_path = matching_files[0]

    # Load required columns
    df = pd.read_csv(
        file_path,
        sep=None,
        engine="python",
        usecols=["time_stamp", "train_test"] + sensor_cols
    )

    # Select the requested data split
    df = df[df["train_test"] == split].copy()

    return df.drop(columns="train_test")

event71_pred = load_sensor_data(71, "prediction")
event24_train = load_sensor_data(24, "train")

# Match measurements by timestamp
matched = event71_pred.merge(
    event24_train,
    on="time_stamp",
    suffixes=("_71", "_24"),
    validate="one_to_one"
)

# Compare sensor values
results = []

for sensor in sensor_cols:

    values_71 = matched[f"{sensor}_71"]
    values_24 = matched[f"{sensor}_24"]

    equal = np.isclose(
        values_71,
        values_24,
        equal_nan=True
    )

    results.append({
        "sensor": sensor,
        "matching_values": equal.sum(),
        "compared_rows": len(equal)
    })

display(pd.DataFrame(results))

,sensor,matching_values,compared_rows
0,sensor_13_avg,72,2449
1,sensor_14_avg,65,2449
2,sensor_18_avg,8,2449


## 7. Inspection of Sensor Value Differences

In [7]:
# Compare sensor values at identical timestamps

comparison_columns = [
    "time_stamp",
    "sensor_13_avg_71",
    "sensor_13_avg_24",
    "sensor_18_avg_71",
    "sensor_18_avg_24"
]

display(
    matched[comparison_columns].head(10)
)

,time_stamp,sensor_13_avg_71,sensor_13_avg_24,sensor_18_avg_71,sensor_18_avg_24
0,2023-01-01 00:00:00,41.0,37.0,1575.9,1336.5
1,2023-01-01 00:10:00,41.0,37.0,1572.7,1307.9
2,2023-01-01 00:20:00,42.0,37.0,1599.2,1325.9
3,2023-01-01 00:30:00,42.0,37.0,1659.8,1322.4
4,2023-01-01 00:40:00,43.0,37.0,1662.9,1292.4
5,2023-01-01 00:50:00,44.0,37.0,1650.5,1259.4
6,2023-01-01 01:00:00,45.0,37.0,1670.7,1297.0
7,2023-01-01 01:10:00,46.0,37.0,1663.6,1299.3
8,2023-01-01 01:20:00,47.0,36.0,1672.6,1354.5
9,2023-01-01 01:30:00,47.0,36.0,1658.3,1308.2


## 8. Training Data Status Distribution

In [8]:
# Inspect operating status distribution in Event 71

file_path = list(farm_dir.rglob("71.csv"))[0]

df = pd.read_csv(
    file_path,
    sep=None,
    engine="python",
    usecols=["train_test", "status_type_id"]
)

status_summary = (
    df.groupby(["train_test", "status_type_id"])
      .size()
      .reset_index(name="observations")
)

display(status_summary)

,train_test,status_type_id,observations
0,prediction,0,2449
1,train,0,44950
2,train,3,1297
3,train,4,6048


## 9. Healthy Training Data Quality

In [9]:
# Load Event 71 with selected sensor features

sensor_cols = [
    "sensor_13_avg",
    "sensor_14_avg",
    "sensor_18_avg"
]

file_path = list(farm_dir.rglob("71.csv"))[0]

data = pd.read_csv(
    file_path,
    sep=None,
    engine="python",
    usecols=[
        "train_test",
        "status_type_id"
    ] + sensor_cols
)

# Select potential healthy training observations
healthy_train = data[
    (data["train_test"] == "train") &
    (data["status_type_id"] == 0)
].copy()

print("Selected training observations:", len(healthy_train))

# Check missing sensor values
missing_summary = pd.DataFrame({
    "missing_values": healthy_train[sensor_cols].isna().sum(),
    "missing_percent": (
        healthy_train[sensor_cols].isna().mean() * 100
    ).round(2)
})

display(missing_summary)

Selected training observations: 44950


,missing_values,missing_percent
sensor_13_avg,0,0.0
sensor_14_avg,0,0.0
sensor_18_avg,0,0.0


## 10. Sensor Plausibility and Statistical Summary

In [10]:
# Calculate descriptive statistics for selected sensors

sensor_statistics = healthy_train[sensor_cols].describe(
    percentiles=[0.01, 0.50, 0.99]
).T

# Select relevant statistics
sensor_statistics = sensor_statistics[
    ["min", "1%", "50%", "99%", "max", "std"]
]

display(sensor_statistics.round(2))

,min,1%,50%,99%,max,std
sensor_13_avg,21.0,23.0,41.0,78.0,86.0,13.82
sensor_14_avg,19.0,21.0,41.0,78.0,90.0,13.40
sensor_18_avg,0.0,0.0,1272.5,1681.3,1682.8,610.04


## 11. Generator Speed Operating Conditions

In [11]:
# Analyze zero and positive generator speed

speed = healthy_train["sensor_18_avg"]

zero_speed = (speed == 0).sum()
positive_speed = (speed > 0).sum()

summary = pd.DataFrame({
    "condition": [
        "Zero generator speed",
        "Positive generator speed"
    ],
    "observations": [
        zero_speed,
        positive_speed
    ]
})

summary["percentage"] = (
    summary["observations"] / len(healthy_train) * 100
).round(2)

display(summary)

,condition,observations,percentage
0,Zero generator speed,2024,4.5
1,Positive generator speed,42926,95.5


## 12. Zero-Speed Sequence Analysis

In [12]:
# Load timestamps and generator speed for Event 71

file_path = list(farm_dir.rglob("71.csv"))[0]

df = pd.read_csv(
    file_path,
    sep=None,
    engine="python",
    usecols=[
        "time_stamp",
        "train_test",
        "status_type_id",
        "sensor_18_avg"
    ]
)

df["time_stamp"] = pd.to_datetime(df["time_stamp"])

# Select training observations with normal operating status
train_data = df[
    (df["train_test"] == "train") &
    (df["status_type_id"] == 0)
].sort_values("time_stamp").copy()

# Identify consecutive zero-speed measurements
zero_speed = train_data["sensor_18_avg"] == 0

time_gap = (
    train_data["time_stamp"].diff()
    != pd.Timedelta(minutes=10)
)

sequence_id = (
    zero_speed.ne(zero_speed.shift(fill_value=False)) |
    time_gap
).cumsum()

# Summarize zero-speed sequences
zero_sequences = (
    train_data.loc[zero_speed]
    .assign(sequence_id=sequence_id[zero_speed])
    .groupby("sequence_id")
    .agg(
        start=("time_stamp", "min"),
        end=("time_stamp", "max"),
        observations=("time_stamp", "size")
    )
)

print("Number of zero-speed sequences:", len(zero_sequences))

print("\nFive longest zero-speed sequences:")
display(
    zero_sequences.nlargest(5, "observations")
    .reset_index(drop=True)
)

Number of zero-speed sequences: 425

Five longest zero-speed sequences:


,start,end,observations
0,2022-03-03 01:50:00,2022-03-03 08:30:00,41
1,2022-10-02 10:00:00,2022-10-02 16:10:00,38
2,2022-03-25 01:50:00,2022-03-25 07:40:00,36
3,2022-10-08 21:50:00,2022-10-09 03:40:00,36
4,2022-09-24 00:10:00,2022-09-24 05:10:00,31


## 13. Wind Speed and Power Feature Identification

In [13]:
# Find feature metadata for Wind Farm A

feature_files = sorted(
    path for path in farm_dir.rglob("*.csv")
    if "feature" in path.name.lower()
)

print("Feature metadata files:", feature_files)

if feature_files:

    features = pd.read_csv(
        feature_files[0],
        sep=None,
        engine="python"
    )

    # Search feature descriptions
    search_text = (
        features.astype(str)
        .agg(" ".join, axis=1)
        .str.lower()
    )

    relevant_features = features[
        search_text.str.contains(
            "wind|power",
            regex=True,
            na=False
        )
    ]

    display(relevant_features.head(30))

else:
    print("No feature metadata file found.")

Feature metadata files: [PosixPath('/content/drive/MyDrive/CARE_To_Compare/Wind Farm A/feature_description.csv')]


,sensor_name,statistics_type,description,unit,is_angle,is_counter
1,sensor_1,average,Wind absolute direction,�,True,False
2,sensor_2,average,Wind relative direction,�,True,False
3,wind_speed_3,"maximum,minimum,average,std_dev",Windspeed,m/s,False,False
4,wind_speed_4,average,Estimated windspeed,m/s,False,False
15,sensor_15,average,Temperature inside generator in stator winding...,�C,False,False
16,sensor_16,average,Temperature inside generator in stator winding...,�C,False,False
17,sensor_17,average,Temperature inside generator in stator winding...,�C,False,False
27,reactive_power_27,"average,maximum,minimum,std_dev",Possible grid capacitive reactive power,kVAr,False,False
28,reactive_power_28,"average,maximum,minimum,std_dev",Possible grid inductive reactive power,kVAr,False,False
29,power_29,"maximum,std_dev,average,minimum",Possible grid active power,kW,False,False


## 14. Wind Speed and Power SCADA Columns

In [14]:
# Inspect available wind speed and power columns

file_path = list(farm_dir.rglob("71.csv"))[0]

# Read column names only
columns = pd.read_csv(
    file_path,
    sep=None,
    engine="python",
    nrows=0
).columns.tolist()

# Relevant sensor prefixes
sensor_prefixes = [
    "wind_speed_3",
    "wind_speed_4",
    "power_29",
    "power_30",
    "sensor_18"
]

for prefix in sensor_prefixes:

    matching_columns = [
        col for col in columns
        if col == prefix or col.startswith(prefix + "_")
    ]

    print(f"\n{prefix}:")
    print(matching_columns)


wind_speed_3:
['wind_speed_3_avg', 'wind_speed_3_max', 'wind_speed_3_min', 'wind_speed_3_std']

wind_speed_4:
['wind_speed_4_avg']

power_29:
['power_29_avg', 'power_29_max', 'power_29_min', 'power_29_std']

power_30:
['power_30_avg', 'power_30_max', 'power_30_min', 'power_30_std']

sensor_18:
['sensor_18_avg', 'sensor_18_max', 'sensor_18_min', 'sensor_18_std']


## 15. Wind Speed and Power by Generator Operating Condition

In [15]:
# Load relevant SCADA features from Event 71

file_path = list(farm_dir.rglob("71.csv"))[0]

columns = [
    "train_test",
    "status_type_id",
    "wind_speed_3_avg",
    "power_30_avg",
    "sensor_18_avg"
]

df = pd.read_csv(
    file_path,
    sep=";",
    usecols=columns
)

# Select training data with normal operating status
normal_train = df[
    (df["train_test"] == "train") &
    (df["status_type_id"] == 0)
].copy()

# Group observations by generator speed
normal_train["speed_group"] = (
    normal_train["sensor_18_avg"]
    .eq(0)
    .map({
        True: "Zero speed",
        False: "Positive speed"
    })
)

# Compare wind speed and grid power
summary = normal_train.groupby("speed_group").agg(
    observations=("sensor_18_avg", "size"),
    wind_median=("wind_speed_3_avg", "median"),
    wind_p90=("wind_speed_3_avg", lambda x: x.quantile(0.90)),
    power_median=("power_30_avg", "median"),
    missing_wind=("wind_speed_3_avg", lambda x: x.isna().sum()),
    missing_power=("power_30_avg", lambda x: x.isna().sum())
)

display(summary.round(2))

,observations,wind_median,wind_p90,power_median,missing_wind,missing_power
speed_group,,,,,,
Positive speed,42926,5.4,11.2,0.12,0,0
Zero speed,2024,1.4,2.1,-0.00,0,0


## 16. Active Power Sensor Comparison

In [16]:
# Load power sensors and generator speed

file_path = list(farm_dir.rglob("71.csv"))[0]

columns = [
    "train_test",
    "status_type_id",
    "sensor_18_avg",
    "power_29_avg",
    "power_30_avg"
]

df = pd.read_csv(
    file_path,
    sep=";",
    usecols=columns
)

# Select training data with positive generator speed
active_train = df[
    (df["train_test"] == "train") &
    (df["status_type_id"] == 0) &
    (df["sensor_18_avg"] > 0)
].copy()

# Compare power sensor distributions
power_statistics = active_train[
    ["power_29_avg", "power_30_avg"]
].describe(
    percentiles=[0.01, 0.50, 0.90, 0.99]
).T

display(
    power_statistics[
        ["min", "1%", "50%", "90%", "99%", "max"]
    ].round(2)
)

,min,1%,50%,90%,99%,max
power_29_avg,0.00,0.00,0.13,0.91,0.98,0.98
power_30_avg,-0.01,-0.01,0.12,0.88,0.98,0.98


## 17. Power Sensor Correlation Analysis

In [17]:
# Compare power sensor relationships

power_data = active_train[
    ["power_29_avg", "power_30_avg"]
].dropna()

# Calculate Pearson correlation
correlation = power_data[
    "power_29_avg"
].corr(
    power_data["power_30_avg"]
)

# Calculate absolute differences
absolute_difference = (
    power_data["power_29_avg"] -
    power_data["power_30_avg"]
).abs()

print("Compared observations:", len(power_data))
print("Pearson correlation:", round(correlation, 4))
print(
    "Median absolute difference:",
    round(absolute_difference.median(), 4)
)
print(
    "Maximum absolute difference:",
    round(absolute_difference.max(), 4)
)

Compared observations: 42926
Pearson correlation: 0.9343
Median absolute difference: 0.0008
Maximum absolute difference: 0.9895


## 18. Power Sensor Discrepancy Analysis

In [18]:
# Calculate absolute power sensor differences
power_check = active_train[
    ["power_29_avg", "power_30_avg"]
].dropna().copy()

power_check["abs_difference"] = (
    power_check["power_29_avg"] -
    power_check["power_30_avg"]
).abs()

# Count large differences
print("Total observations:", len(power_check))

for threshold in [0.1, 0.5]:
    count = (power_check["abs_difference"] > threshold).sum()
    percentage = count / len(power_check) * 100

    print(
        f"Difference > {threshold}: "
        f"{count} observations ({percentage:.2f}%)"
    )

# Display five largest differences
print("\nFive largest differences:")
display(
    power_check.nlargest(5, "abs_difference").round(4)
)

Total observations: 42926
Difference > 0.1: 970 observations (2.26%)
Difference > 0.5: 706 observations (1.64%)

Five largest differences:


,power_29_avg,power_30_avg,abs_difference
29564,0.9756,-0.0139,0.9895
28464,0.9756,-0.0138,0.9894
28402,0.9755,-0.0138,0.9893
28443,0.9756,-0.0136,0.9892
28543,0.9756,-0.0136,0.9892


## 19. Temporal Train and Validation Split

In [19]:
# Define model features and target
feature_cols = [
    "wind_speed_3_avg",
    "sensor_18_avg",
    "power_30_avg"
]

target_col = "sensor_13_avg"

# Load Event 71
file_path = list(farm_dir.rglob("71.csv"))[0]

df = pd.read_csv(
    file_path,
    sep=";",
    usecols=[
        "time_stamp",
        "train_test",
        "status_type_id"
    ] + feature_cols + [target_col]
)

df["time_stamp"] = pd.to_datetime(df["time_stamp"])

# Select active operating conditions
model_data = df[
    (df["train_test"] == "train") &
    (df["status_type_id"] == 0) &
    (df["sensor_18_avg"] > 0)
].copy()

# Remove rows with missing model values
model_data = model_data.dropna(
    subset=feature_cols + [target_col]
)

# Sort chronologically
model_data = model_data.sort_values(
    "time_stamp"
).reset_index(drop=True)

# Check duplicate timestamps
print(
    "Duplicate timestamps:",
    model_data["time_stamp"].duplicated().sum()
)

# Chronological 80/20 split
split_index = int(len(model_data) * 0.8)

train_split = model_data.iloc[:split_index].copy()
val_split = model_data.iloc[split_index:].copy()

# Display split summary
split_summary = pd.DataFrame({
    "dataset": ["Training", "Validation"],
    "observations": [len(train_split), len(val_split)],
    "start": [
        train_split["time_stamp"].min(),
        val_split["time_stamp"].min()
    ],
    "end": [
        train_split["time_stamp"].max(),
        val_split["time_stamp"].max()
    ]
})

display(split_summary)

Duplicate timestamps: 0


,dataset,observations,start,end
0,Training,34340,2022-01-01 00:00:00,2022-10-31 14:00:00
1,Validation,8586,2022-10-31 14:10:00,2022-12-31 23:50:00


## 20. Baseline Model – Linear Regression

In [20]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

# Prepare training data
X_train = train_split[feature_cols]
y_train = train_split[target_col]

# Prepare validation data
X_val = val_split[feature_cols]
y_val = val_split[target_col]

# Initialize the baseline model
model = LinearRegression()

# Train on training data only
model.fit(X_train, y_train)

# Generate temperature predictions
train_predictions = model.predict(X_train)
val_predictions = model.predict(X_val)

# Evaluate prediction errors
train_mae = mean_absolute_error(
    y_train, train_predictions
)

val_mae = mean_absolute_error(
    y_val, val_predictions
)

print("Baseline Model: Linear Regression")
print(f"Training MAE: {train_mae:.3f}")
print(f"Validation MAE: {val_mae:.3f}")

Baseline Model: Linear Regression
Training MAE: 6.050
Validation MAE: 5.463


## 21. Dummy Baseline Comparison

In [21]:
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error

# Create a simple constant prediction model
dummy_model = DummyRegressor(strategy="median")

# Fit using training data only
dummy_model.fit(X_train, y_train)

# Generate predictions
dummy_train_pred = dummy_model.predict(X_train)
dummy_val_pred = dummy_model.predict(X_val)

# Calculate errors
dummy_train_mae = mean_absolute_error(
    y_train, dummy_train_pred
)

dummy_val_mae = mean_absolute_error(
    y_val, dummy_val_pred
)

# Compare both models
results = pd.DataFrame({
    "Model": [
        "Dummy Baseline",
        "Linear Regression"
    ],
    "Training MAE": [
        dummy_train_mae,
        train_mae
    ],
    "Validation MAE": [
        dummy_val_mae,
        val_mae
    ]
})

display(results.round(3))

,Model,Training MAE,Validation MAE
0,Dummy Baseline,11.212,10.488
1,Linear Regression,6.050,5.463


## 22. Residual Analysis

In [22]:
# Calculate residuals
train_residuals = y_train.to_numpy() - train_predictions
val_residuals = y_val.to_numpy() - val_predictions

# Summarize residual distributions
residual_summary = pd.DataFrame({
    "dataset": ["Training", "Validation"],
    "mean_residual": [
        train_residuals.mean(),
        val_residuals.mean()
    ],
    "median_residual": [
        pd.Series(train_residuals).median(),
        pd.Series(val_residuals).median()
    ],
    "p95_absolute_residual": [
        pd.Series(abs(train_residuals)).quantile(0.95),
        pd.Series(abs(val_residuals)).quantile(0.95)
    ],
    "max_absolute_residual": [
        abs(train_residuals).max(),
        abs(val_residuals).max()
    ]
})

display(residual_summary.round(2))

,dataset,mean_residual,median_residual,p95_absolute_residual,max_absolute_residual
0,Training,-0.00,-0.65,14.68,39.46
1,Validation,-2.83,-3.26,13.09,24.15


## 23. Monthly Residual Analysis

In [23]:
# Create validation residual dataset
residual_data = val_split[["time_stamp"]].copy()

residual_data["residual"] = (
    y_val.to_numpy() - val_predictions
)

# Extract month from timestamp
residual_data["month"] = (
    residual_data["time_stamp"]
    .dt.to_period("M")
    .astype(str)
)

# Calculate monthly residual statistics
monthly_summary = residual_data.groupby("month").agg(
    observations=("residual", "size"),
    mean_residual=("residual", "mean"),
    median_residual=("residual", "median"),
    mae=("residual", lambda x: x.abs().mean())
)

display(monthly_summary.round(2))

,observations,mean_residual,median_residual,mae
month,,,,
2022-10,59,6.83,7.06,6.97
2022-11,4125,-2.38,-2.82,5.13
2022-12,4402,-3.38,-3.75,5.76


## 24. Initial Anomaly Threshold

In [24]:
import numpy as np

# Define an initial threshold using training residuals
threshold = np.quantile(train_residuals, 0.99)

# Identify positive residuals above the threshold
train_alerts = train_residuals > threshold
val_alerts = val_residuals > threshold

print(f"Initial anomaly threshold: {threshold:.2f}")

print(
    f"Training alerts: {train_alerts.sum()} "
    f"({train_alerts.mean() * 100:.2f}%)"
)

print(
    f"Validation alerts: {val_alerts.sum()} "
    f"({val_alerts.mean() * 100:.2f}%)"
)

Initial anomaly threshold: 18.80
Training alerts: 344 (1.00%)
Validation alerts: 0 (0.00%)


## 25. Positive Residual Distribution

In [25]:
# Compare positive residual distributions

residual_comparison = pd.DataFrame({
    "dataset": ["Training", "Validation"],
    "positive_residual_percent": [
        (train_residuals > 0).mean() * 100,
        (val_residuals > 0).mean() * 100
    ],
    "p95_residual": [
        np.quantile(train_residuals, 0.95),
        np.quantile(val_residuals, 0.95)
    ],
    "p99_residual": [
        np.quantile(train_residuals, 0.99),
        np.quantile(val_residuals, 0.99)
    ],
    "max_residual": [
        train_residuals.max(),
        val_residuals.max()
    ]
})

display(residual_comparison.round(2))

,dataset,positive_residual_percent,p95_residual,p99_residual,max_residual
0,Training,46.69,13.10,18.80,32.83
1,Validation,29.94,7.84,11.65,18.37


## 26. Anomaly Event – Pre-Event Data Inspection

In [26]:
# Load Event 0 – Generator Bearing Failure
file_path = list(farm_dir.rglob("0.csv"))[0]

columns = [
    "time_stamp",
    "train_test",
    "status_type_id"
] + feature_cols + [target_col]

event0 = pd.read_csv(
    file_path,
    sep=";",
    usecols=columns
)

event0["time_stamp"] = pd.to_datetime(
    event0["time_stamp"]
)

# Get documented event start from metadata
event_start = pd.to_datetime(
    events.loc[
        events["event_id"] == 0,
        "event_start"
    ].iloc[0]
)

# Select prediction period
prediction_data = event0[
    event0["train_test"] == "prediction"
].copy()

# Select observations before documented event start
pre_event = prediction_data[
    prediction_data["time_stamp"] < event_start
].copy()

# Select active operating conditions
active_pre_event = pre_event[
    (pre_event["status_type_id"] == 0) &
    (pre_event["sensor_18_avg"] > 0)
].copy()

print("Documented event start:", event_start)
print("Total prediction observations:", len(prediction_data))
print("Pre-event observations:", len(pre_event))
print("Active pre-event observations:", len(active_pre_event))

print(
    "Rows with missing model values:",
    active_pre_event[
        feature_cols + [target_col]
    ].isna().any(axis=1).sum()
)

Documented event start: 2023-08-06 06:10:00
Total prediction observations: 2838
Pre-event observations: 288
Active pre-event observations: 248
Rows with missing model values: 0


## 27. First Anomaly Event Evaluation

In [27]:
# Prepare pre-event data
X_event0 = active_pre_event[feature_cols]
y_event0 = active_pre_event[target_col]

# Predict bearing temperature
event0_predictions = model.predict(X_event0)

# Calculate residuals
event0_residuals = (
    y_event0.to_numpy() - event0_predictions
)

# Detect anomalies using the existing threshold
event0_alerts = event0_residuals > threshold

# Display evaluation results
print("Event 0 – Generator Bearing Failure")
print("Pre-event observations:", len(event0_residuals))

print(
    "Mean residual:",
    round(event0_residuals.mean(), 2)
)

print(
    "Maximum positive residual:",
    round(event0_residuals.max(), 2)
)

print(
    "Anomaly threshold:",
    round(threshold, 2)
)

print(
    "Pre-event alerts:",
    event0_alerts.sum()
)

print(
    "Alert percentage:",
    round(event0_alerts.mean() * 100, 2), "%"
)

Event 0 – Generator Bearing Failure
Pre-event observations: 248
Mean residual: 10.28
Maximum positive residual: 23.76
Anomaly threshold: 18.8
Pre-event alerts: 15
Alert percentage: 6.05 %


## 28. First Alert and Lead Time Analysis

In [28]:
# Create detailed pre-event prediction results
alert_details = active_pre_event[
    ["time_stamp", target_col]
].copy()

alert_details["predicted_temperature"] = event0_predictions
alert_details["residual"] = event0_residuals
alert_details["alert"] = event0_alerts

# Select detected alerts
detected_alerts = alert_details[
    alert_details["alert"]
].sort_values("time_stamp").copy()

# Calculate time before documented event start
detected_alerts["lead_time_hours"] = (
    event_start - detected_alerts["time_stamp"]
).dt.total_seconds() / 3600

print("Documented event start:", event_start)
print("Total alerts:", len(detected_alerts))

if not detected_alerts.empty:
    first_alert = detected_alerts.iloc[0]

    print("First alert:", first_alert["time_stamp"])
    print(
        "First alert lead time (hours):",
        round(first_alert["lead_time_hours"], 2)
    )

    display(detected_alerts.round(2))
else:
    print("No pre-event alerts detected.")

Documented event start: 2023-08-06 06:10:00
Total alerts: 15
First alert: 2023-08-04 17:40:00
First alert lead time (hours): 36.5


,time_stamp,sensor_13_avg,predicted_temperature,residual,alert,lead_time_hours
52217,2023-08-04 17:40:00,58.0,35.79,22.21,True,36.50
52218,2023-08-04 17:50:00,57.0,36.92,20.08,True,36.33
52219,2023-08-04 18:00:00,57.0,34.46,22.54,True,36.17
52222,2023-08-04 18:30:00,58.0,38.43,19.57,True,35.67
52228,2023-08-04 19:30:00,60.0,40.33,19.67,True,34.67
52229,2023-08-04 19:40:00,60.0,39.35,20.65,True,34.50
52230,2023-08-04 19:50:00,60.0,39.37,20.63,True,34.33
52231,2023-08-04 20:00:00,59.0,37.18,21.82,True,34.17
52232,2023-08-04 20:10:00,59.0,35.24,23.76,True,34.00
52233,2023-08-04 20:20:00,58.0,36.18,21.82,True,33.83


## 29. Normal Event Control Evaluation

In [29]:
# Load Event 24 – Normal Control Event
file_path = list(farm_dir.rglob("24.csv"))[0]

columns = [
    "train_test",
    "status_type_id"
] + feature_cols + [target_col]

event24 = pd.read_csv(
    file_path,
    sep=";",
    usecols=columns
)

# Select normal prediction observations
normal_control = event24[
    (event24["train_test"] == "prediction") &
    (event24["status_type_id"] == 0) &
    (event24["sensor_18_avg"] > 0)
].copy()

# Remove rows with missing model values
normal_control = normal_control.dropna(
    subset=feature_cols + [target_col]
)

# Apply the existing trained model
control_predictions = model.predict(
    normal_control[feature_cols]
)

# Calculate residuals and alerts
control_residuals = (
    normal_control[target_col].to_numpy()
    - control_predictions
)

control_alerts = control_residuals > threshold

print("Event 24 – Normal Control")
print("Evaluated observations:", len(normal_control))
print("Mean residual:", round(control_residuals.mean(), 2))
print("Maximum positive residual:", round(control_residuals.max(), 2))
print("Anomaly threshold:", round(threshold, 2))
print("Alerts:", control_alerts.sum())

print(
    "Alert percentage:",
    round(control_alerts.mean() * 100, 2), "%"
)

Event 24 – Normal Control
Evaluated observations: 2658
Mean residual: -1.74
Maximum positive residual: 20.17
Anomaly threshold: 18.8
Alerts: 3
Alert percentage: 0.11 %


## 30. Normal Event Alert Inspection

In [30]:
# Load timestamps for Event 24
timestamps = pd.read_csv(
    file_path,
    sep=";",
    usecols=["time_stamp"]
)

# Select the three detected alerts
normal_alert_details = normal_control.loc[
    control_alerts,
    [target_col]
].copy()

# Add timestamps
normal_alert_details["time_stamp"] = pd.to_datetime(
    timestamps.loc[
        normal_alert_details.index,
        "time_stamp"
    ]
)

# Add predictions and residuals
normal_alert_details["predicted_temperature"] = (
    control_predictions[control_alerts]
)

normal_alert_details["residual"] = (
    control_residuals[control_alerts]
)

# Display alerts chronologically
display(
    normal_alert_details[
        [
            "time_stamp",
            target_col,
            "predicted_temperature",
            "residual"
        ]
    ].sort_values("time_stamp").round(2)
)

,time_stamp,sensor_13_avg,predicted_temperature,residual
53506,2023-05-03 02:00:00,68.0,49.09,18.91
53622,2023-05-03 21:20:00,71.0,51.09,19.91
53623,2023-05-03 21:30:00,68.0,47.83,20.17


## 31. Key Findings and Conclusions

### Dataset Selection

- Selected Wind Farm A as the initial experimental dataset.
- Selected Event 71 as the normal-behavior training candidate.
- Investigated SCADA operating conditions, sensor completeness, and event-specific data structures.
- Applied a chronological train/validation split within Event 71.

### Baseline Model

A Linear Regression model was trained to predict generator bearing temperature (`sensor_13_avg`) using:

- `wind_speed_3_avg`
- `sensor_18_avg`
- `power_30_avg`

**Model performance:**

| Model | Validation MAE |
|---|---:|
| Dummy Baseline | 10.488 |
| Linear Regression | 5.463 |

The Linear Regression model achieved approximately 47.9% lower validation MAE than the Dummy Baseline.

### Anomaly Detection

An initial positive-residual threshold of 18.80 was established using the 99th percentile of training residuals.

**Exploratory evaluation:**

| Metric | Event 0 (Anomaly) | Event 24 (Normal) |
|---|---:|---:|
| Evaluated observations | 248 | 2,658 |
| Alerts | 15 | 3 |
| Alert rate | 6.05% | 0.11% |

The earliest alert in Event 0 occurred 36.5 hours before the documented generator bearing failure event start.

The three alerts in Event 24 formed two temporally separated alert groups.

### Limitations

- Only one anomaly event and one normal control event were evaluated.
- Event-specific anonymization limits cross-event chronological interpretation.
- Operating conditions may differ between event datasets.
- The threshold is preliminary and has not been independently calibrated.
- Individual alerts do not prove physical failure or successful early warning.
- The documented event start may differ from the physical onset of a fault.

### Conclusion

The initial Normal Behavior Model demonstrates promising temperature prediction performance and produces elevated residuals before a documented generator bearing failure.

However, the current results are exploratory and do not establish reliable predictive maintenance performance.

### Next Steps

1. Evaluate additional normal and anomaly events.
2. Introduce event-based alert evaluation.
3. Measure false alarms and detection lead times.
4. Investigate threshold calibration and model generalization.
5. Compare the baseline against more advanced approaches.